# 🫀 실험 14 — 임상 동작점: **민감도를 먼저 고정하면 무엇을 잃는가**

**MedKOS / `notebooks/exp14_clinical_operating_point.ipynb`** · 퀘스트 `ailab-2026-0015`
**학습 0회** — 실험10′의 out-of-fold 확률을 다시 자르기만 한다.

---

## ⚠️ 먼저 읽을 것 — 이 데이터로는 **급성** 심근경색을 못 잰다

퀘스트 요구사항 **R1**은 "급성 관상동맥증후군은 위음성이 지배하니 민감도 0.90~0.95를
먼저 고정하라"였다. 그런데 **PTB-XL의 `MI` superclass는 대부분 '경색 패턴'(병적 Q파 등)이고
급성 여부를 구분하지 않는다.** 급성 손상 소견은 오히려 `STTC`(ISCA/ISCI 등) 쪽에 섞여 있다.

> **그러므로 이 실험이 답하는 것은 "급성 MI 배제(rule-out) 성능"이 아니라
> "MI 패턴 검출을 재현율 우선으로 돌리면 특이도가 어디까지 버티는가"다.**
> R1은 여기서 **예행연습**만 할 수 있고, 진짜 답은 응급실 코호트(내원시각·트로포닌이 붙은
> MIMIC-IV-ECG 계열)에서만 나온다. 그건 큐 6번의 데이터다.

그래도 지금 할 값어치가 있다. **R1이 현실적인 요구인지**, 그리고 **웨어러블 구성이 그 제약을
견디는지**를 재학습 없이 미리 알 수 있기 때문이다. 실험15의 손실함수 설계가 여기에 달려 있다.

## 무엇을 재나

MI를 **일대다(one-vs-rest)** 로 놓고 확률 임계값을 훑는다. 음성군은 NORM만이 아니라
**MI가 아닌 전부**(NORM·CD·STTC·HYP)다 — 응급실에는 정상인만 오지 않는다.

각 유도 구성에서:
- **민감도 0.90 / 0.95를 고정**했을 때의 특이도 · 경보율 · PPV · NPV
- **NPV는 유병률의 함수**이므로 가정 유병률별로 표를 낸다 (배제 도구의 핵심 지표)
- `{II}` · `{I,II}` · `{II,V1}` · `{12}` 를 나란히 → 웨어러블이 이 제약을 견디는가

## 임계값을 고르는 법 — **교차적합**이 아니면 낙관 편향이 낀다

`F_β = (1+β²)·P·R / (β²·P + R)` 를 최대화하는 임계값을 쓰되, **고른 데이터에서 평가하면 안 된다.**

실험10′의 산출물은 겹별 out-of-fold 확률이므로 이렇게 한다:

> **겹 k의 임계값은 겹 k를 뺀 나머지에서 고르고, 겹 k에만 적용한다.**
> 그렇게 얻은 겹별 예측을 모아 전체 민감도·특이도를 낸다.

그리고 **전체 데이터에서 고른 임계값**(낙관적)도 같이 내서 **낙관 편차(optimism gap)** 를
수치로 보여준다. 이 편차가 크면 앞으로 어떤 임계값 보고도 교차적합 없이는 못 믿는다.

## 사전등록 (결과 보기 전에 고정)

| | 예측 | 근거 |
|---|---|---|
| **G0** | 교차적합 임계값이 검정 겹에서 목표 민감도 ±0.05 안에 착지 | 임계값이 겹 사이를 건너가는가(전이 가능성) |
| **P-1 ★** | `{12}`에서 민감도 0.90일 때 **특이도 ≥ 0.60** | 임상 수용 가능 하한. 못 넘으면 동작점 조정만으론 부족하고 실험15에서 손실함수를 손대야 한다 |
| **P-2** | 민감도 0.90에서 `{I,II}`의 특이도가 `{12}` 대비 **−0.10 이내** | 웨어러블이 재현율 우선 제약을 견디는가 |
| **P-3** | 낙관 편차(전체선택 − 교차적합)의 민감도 차 **< 0.05** | 편차가 크면 이후 모든 임계값 보고에 교차적합을 의무화한다 |

**P-1이 실패하는 것도 결과다.** "이 모델·이 데이터로는 민감도 0.90을 쓸 수 없다"가
실험15의 설계 입력이 된다(가중 손실 → Focal 순서로).


In [ ]:
# CELL 1 — 설정 (학습 없음)
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

CLASSES = ["NORM", "CD", "STTC", "MI", "HYP"]
CONFIGS = {"II": [1], "I+II": [0, 1], "II+V1": [1, 6], "12": list(range(12))}
K_FOLD, SEED0, BOOT = 5, 20260801, 4000

TARGETS   = [0.90, 0.95]                 # 고정할 민감도
BETAS     = [2, 5]                       # F-beta 의 beta
PREVS     = [0.02, 0.05, 0.10, 0.20]     # NPV 계산용 가정 유병률
SPEC_FLOOR = 0.60                        # P-1 의 임상 수용 하한
PANELS = {"MI": "MI", "STTC": "STTC"}    # 주패널 MI · 부패널 STTC(급성 손상이 섞인 쪽)

CONFIG = dict(exp="exp14_clinical_operating_point", quest="ailab-2026-0015",
              parent_exp="exp10p_lead_cv", training="없음(재분석)",
              purpose="R1(재현율 우선 동작점)이 현실적인 요구인지 재학습 없이 미리 잰다",
              caveat=("PTB-XL의 MI는 대부분 '경색 패턴'이고 급성 여부를 구분하지 않는다. "
                      "급성 rule-out 성능이 아니라 MI 패턴 검출의 재현율-특이도 교환을 잰다"),
              threshold_selection="F-beta 를 겹 k 제외 데이터에서 선택 → 겹 k 에 적용(교차적합)",
              predictions={"G0": "교차적합 임계값이 검정 겹에서 목표 민감도 ±0.05",
                           "P-1": f"{{12}} 민감도 0.90에서 특이도 >= {SPEC_FLOOR}",
                           "P-2": "민감도 0.90에서 {I,II} 특이도가 {12} 대비 -0.10 이내",
                           "P-3": "낙관 편차의 민감도 차 < 0.05"},
              targets=TARGETS, betas=BETAS, prevalences=PREVS,
              k_fold=K_FOLD, boot=BOOT, seed0=SEED0)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp14_clinical_op", CONFIG, project=PROJECT)

REG = os.path.join(PROJECT, "registry.jsonl")
prev = None
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") == "exp10p_lead_cv" and os.path.isdir(r.get("dir", "")):
        prev = r
if prev is None:
    raise RuntimeError("registry.jsonl에서 exp10p_lead_cv를 못 찾았습니다")
PREV = prev["dir"]
run.log(f"실험10′ 산출물: {PREV}")

def prev_arm(name):
    p = os.path.join(PREV, "arms", name, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

missing = [f"fixed_{c}_f{k}" for c in CONFIGS for k in range(K_FOLD)
           if prev_arm(f"fixed_{c}_f{k}") is None]
if missing:
    raise RuntimeError(f"실험10′ arm 없음: {missing[:5]} …")
run.log("✅ 유도고정 arm 확인 — 학습하지 않습니다")

In [ ]:
# CELL 2 — 라벨 + OOF (X 미사용)
import pandas as pd, subprocess

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
for f in ("ptbxl_database.csv", "scp_statements.csv"):
    d = os.path.join(PTB, f)
    if not (os.path.exists(d) and os.path.getsize(d) > 0):
        subprocess.run(["wget", "-q", "-O", d, f"{BASE}/{f}"])
df  = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")
scp = pd.read_csv(os.path.join(PTB, "scp_statements.csv"), index_col=0)
agg = scp[scp.diagnostic == 1].diagnostic_class.to_dict()
df["sc"] = df.scp_codes.apply(
    lambda s: sorted({agg[k] for k in ast.literal_eval(s) if k in agg}))
sub = df[df.sc.apply(lambda s: len(s) == 1 and s[0] in CLASSES)].copy()

CACHE = run.data("ptbxl_12lead_full.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"캐시가 없습니다: {CACHE}")
z = np.load(CACHE, allow_pickle=True)
Y, FOLD10, EID = z["y"], z["fold"], z["eid"]
if len(Y) != len(sub) or not np.array_equal(np.sort(EID), np.sort(sub.index.values)):
    raise RuntimeError("캐시가 지금 sub와 다르다")
CV = (FOLD10 - 1) % K_FOLD

OOF = {}
for c in CONFIGS:
    arr = np.zeros((len(Y), len(CLASSES)))
    for k in range(K_FOLD):
        arr[np.where(CV == k)[0]] = prev_arm(f"fixed_{c}_f{k}")
    OOF[c] = arr
run.log(f"레코드 {len(Y):,} · 클래스 {np.bincount(Y, minlength=5).tolist()} ({CLASSES})")
for p, cls in PANELS.items():
    i = CLASSES.index(cls)
    run.log(f"  {p}: 양성 {int((Y==i).sum()):,} / 음성 {int((Y!=i).sum()):,} "
            f"(데이터셋 유병률 {(Y==i).mean():.3f})")

### CELL 3 — 민감도를 고정했을 때의 대가

민감도 목표를 고정하고 임계값을 거기 맞춘 뒤 특이도를 읽는다.
**부트스트랩 안에서도 임계값을 다시 맞춘다** — 임계값 선택의 잡음까지 CI에 넣기 위해서다.

- **경보율** = 전체 중 양성으로 찍히는 비율. 현장 업무량이 이 숫자다.
- **PPV/NPV**는 유병률에 의존하므로 CELL 5에서 가정 유병률별로 따로 낸다.

> **'실민감도'가 목표보다 크게 높으면 확률에 동점(예: 0으로 눌린 값)이 많다는 뜻**이다.
> 그 경우 임계값이 요구한 것보다 엄격한 지점에 앉으므로 **보고된 특이도는 보수적**(실제보다
> 나쁘게)이다. 과잉 달성이 0.05를 넘으면 **G0가 실패**하도록 해뒀다 — 표를 그냥 믿지 말고
> G0를 먼저 볼 것.


In [ ]:
# CELL 3 — 민감도 고정 → 특이도 · 경보율
def spec_at_sens(score, pos, target):
    """민감도 >= target 을 만족하는 가장 높은 임계값에서의 (임계값, 민감도, 특이도, 경보율)."""
    p = score[pos]; n = score[~pos]
    if len(p) == 0 or len(n) == 0:
        return np.nan, np.nan, np.nan, np.nan
    # 양성의 하위 (1-target) 분위 = 그 값 이상이면 target 만큼 잡힌다
    thr = float(np.quantile(p, 1.0 - target, method="lower"))
    sens = float((p >= thr).mean())
    spec = float((n < thr).mean())
    alarm = float((score >= thr).mean())
    return thr, sens, spec, alarm

# ★ 재표본 인덱스를 미리 만들어두면 4000 x 16k x 8B = 520MB 다 — 스트림을 매번 다시 돌린다.
#   같은 시드에서 같은 순서로 뽑으므로 모든 구성이 동일한 재표본 축을 공유한다(짝지은 비교).
def boot_stream():
    rs = np.random.RandomState(SEED0)
    for _ in range(BOOT):
        yield rs.randint(0, len(Y), len(Y))

KEY = lambda t, c: f"{t:.2f}|{c}"

OP = {}
for pname, cls in PANELS.items():
    ci_ = CLASSES.index(cls); pos = (Y == ci_)
    OP[pname] = {}
    run.log("\n" + "=" * 104)
    run.log(f"【{pname}】 민감도를 고정했을 때의 특이도 (일대다 · 음성군 = {pname} 아닌 전부)")
    run.log("=" * 104)
    run.log(f"  {'목표민감도':<10}{'구성':<8}{'임계값':>9}{'실민감도':>10}{'특이도':>10}"
            f"{'  특이도 95% CI':<22}{'경보율':>9}")
    for t in TARGETS:
        for c in CONFIGS:
            s = OOF[c][:, ci_]
            thr, sens, spec, alarm = spec_at_sens(s, pos, t)
            bs = np.empty(BOOT)
            for b, idx in enumerate(boot_stream()):
                _, _, sp, _ = spec_at_sens(s[idx], pos[idx], t)
                bs[b] = sp
            lo, hi = float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))
            OP[pname][KEY(t, c)] = {"target": t, "config": c, "thr": thr,
                                     "sens": sens, "spec": spec, "spec_ci": [lo, hi],
                                     "alarm": alarm}
            run.log(f"  {t:<10.2f}{c:<8}{thr:>9.4f}{sens:>10.3f}{spec:>10.3f}"
                    f"  [{lo:.3f}, {hi:.3f}]      {alarm:>9.3f}")
        run.log("  " + "-" * 100)

In [ ]:
# CELL 4 — 교차적합 F-beta 임계값 + 낙관 편차
def fbeta_at(score, pos, thr, beta):
    pred = score >= thr
    tp = float((pred & pos).sum()); fp = float((pred & ~pos).sum()); fn = float((~pred & pos).sum())
    if tp == 0: return 0.0
    prec, rec = tp / (tp + fp), tp / (tp + fn)
    b2 = beta * beta
    return (1 + b2) * prec * rec / (b2 * prec + rec)

def best_thr(score, pos, beta, grid=400):
    qs = np.quantile(score, np.linspace(0.001, 0.999, grid))
    vals = [fbeta_at(score, pos, t, beta) for t in qs]
    return float(qs[int(np.argmax(vals))]), float(np.max(vals))

FB = {}
for pname, cls in PANELS.items():
    ci_ = CLASSES.index(cls); pos = (Y == ci_)
    FB[pname] = {}
    run.log("\n" + "=" * 110)
    run.log(f"【{pname}】 F-β 임계값 — 교차적합 vs 전체선택(낙관)")
    run.log("=" * 110)
    run.log(f"  {'β':<4}{'구성':<8}"
            f"{'교차 민감도':>12}{'교차 특이도':>12}"
            f"{'낙관 민감도':>12}{'낙관 특이도':>12}{'  낙관편차(민감도)':>18}")
    for beta in BETAS:
        for c in CONFIGS:
            s = OOF[c][:, ci_]
            # ── 교차적합: 겹 k의 임계값은 겹 k를 뺀 데이터에서 고른다
            pred = np.zeros(len(Y), bool); thrs = []
            for k in range(K_FOLD):
                te = (CV == k); tr = ~te
                th, _ = best_thr(s[tr], pos[tr], beta)
                thrs.append(th); pred[te] = s[te] >= th
            cv_sens = float(pred[pos].mean()); cv_spec = float((~pred[~pos]).mean())
            # ── 낙관: 전체에서 고르고 전체에서 평가
            th_all, _ = best_thr(s, pos, beta)
            pa = s >= th_all
            op_sens = float(pa[pos].mean()); op_spec = float((~pa[~pos]).mean())
            FB[pname][f"b{beta}|{c}"] = {
                "beta": beta, "config": c, "fold_thr": thrs,
                "cv": {"sens": cv_sens, "spec": cv_spec},
                "optimistic": {"sens": op_sens, "spec": op_spec, "thr": th_all},
                "optimism_sens": op_sens - cv_sens, "optimism_spec": op_spec - cv_spec}
            run.log(f"  {beta:<4}{c:<8}{cv_sens:>12.3f}{cv_spec:>12.3f}"
                    f"{op_sens:>12.3f}{op_spec:>12.3f}{op_sens - cv_sens:>+18.4f}")
        run.log("  " + "-" * 106)

In [ ]:
# CELL 5 — NPV는 유병률의 함수다 (배제 도구의 핵심 지표)
def ppv_npv(sens, spec, prev):
    tp = sens * prev; fn = (1 - sens) * prev
    tn = spec * (1 - prev); fp = (1 - spec) * (1 - prev)
    return (tp / (tp + fp) if tp + fp > 0 else float("nan"),
            tn / (tn + fn) if tn + fn > 0 else float("nan"))

NPV = {}
for pname in PANELS:
    run.log("\n" + "=" * 104)
    run.log(f"【{pname}】 가정 유병률별 PPV / NPV  (민감도 고정 동작점 기준)")
    run.log("=" * 104)
    run.log(f"  {'목표민감도':<10}{'구성':<8}" + "".join(f"{'p=' + f'{p:.0%}':>18}" for p in PREVS))
    for t in TARGETS:
        for c in CONFIGS:
            m = OP[pname][KEY(t, c)]
            cells = []
            for p in PREVS:
                ppv, npv = ppv_npv(m["sens"], m["spec"], p)
                NPV[f"{pname}|{t}|{c}|{p}"] = {"ppv": ppv, "npv": npv}
                cells.append(f"{ppv:>7.3f}/{npv:<9.3f}")
            run.log(f"  {t:<10.2f}{c:<8}" + "".join(f"{x:>18}" for x in cells))
        run.log("  " + "-" * 100)
    run.log("  (칸은 PPV/NPV · 데이터셋 자체 유병률이 아니라 **가정** 유병률이다)")

# ── 사전등록 채점
MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}
mi = OP["MI"]
g0_ok = all(abs(mi[KEY(t, c)]["sens"] - t) <= 0.05 for t in TARGETS for c in CONFIGS)
p1_m = mi[KEY(0.90, "12")]
# 3분: CI 하한이 기준선을 넘으면 지지 / CI 상한이 못 미치면 기각 / 걸치면 미결
P1 = (True if p1_m["spec_ci"][0] >= SPEC_FLOOR
      else (False if p1_m["spec_ci"][1] < SPEC_FLOOR else None))
gap = mi[KEY(0.90, "I+II")]["spec"] - mi[KEY(0.90, "12")]["spec"]
P2 = bool(gap >= -0.10)
opt = max(abs(v["optimism_sens"]) for v in FB["MI"].values())
P3 = bool(opt < 0.05)

run.log("\n" + "=" * 104)
run.log("【사전등록 채점】")
run.log("=" * 104)
run.log(f"  G0 교차적합 임계값이 목표 민감도 ±0.05 → {'✅' if g0_ok else '❌'}")
run.log(f"  P-1 ★ {{12}} 민감도 0.90에서 특이도 ≥ {SPEC_FLOOR} → {MARK[P1]}  "
        f"(특이도 {p1_m['spec']:.3f} [{p1_m['spec_ci'][0]:.3f}, {p1_m['spec_ci'][1]:.3f}])")
run.log(f"  P-2  {{I,II}}가 {{12}} 대비 −0.10 이내 → {MARK[P2]}  "
        f"({mi[KEY(0.90,'I+II')]['spec']:.3f} vs {mi[KEY(0.90,'12')]['spec']:.3f}, 차 {gap:+.3f})")
run.log(f"  P-3  낙관 편차(민감도) < 0.05 → {MARK[P3]}  (최대 {opt:+.4f})")

if P1 is True and P2:
    verdict = (f"R1 성립 — 민감도 0.90을 고정해도 {{12}} 특이도 {p1_m['spec']:.3f}로 하한을 넘고, "
               f"웨어러블 {{I,II}}도 {gap:+.3f} 안에서 따라온다. **동작점 조정만으로 충분**하며 "
               "실험15에서 손실함수를 손댈 필요가 없다")
elif P1 is False:
    verdict = (f"R1 미달 — 민감도 0.90에서 {{12}} 특이도가 {p1_m['spec']:.3f}로 하한 {SPEC_FLOOR}에 "
               "못 미친다. **동작점 조정만으로는 안 된다** → 실험15에서 가중 손실(pos_weight) → "
               "Focal 순으로 하나씩 적용한다")
else:
    verdict = (f"미결 — {{12}} 특이도 {p1_m['spec']:.3f}의 CI가 하한 {SPEC_FLOOR}을 걸친다. "
               "실험15는 동작점 조정을 먼저 시도하되 손실가중을 대안으로 준비한다")
run.log(f"\n▶ {verdict}")
run.log("=" * 104)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(PANELS), figsize=(6.2 * len(PANELS), 4.2), squeeze=False)
for ax, (pname, cls) in zip(axes[0], PANELS.items()):
    ci_ = CLASSES.index(cls); pos = (Y == ci_)
    for c in CONFIGS:
        s = OOF[c][:, ci_]
        ts = np.linspace(0.50, 0.995, 120)
        sp = [spec_at_sens(s, pos, t)[2] for t in ts]
        ax.plot(ts, sp, label=c, lw=1.8)
    ax.axhline(SPEC_FLOOR, ls="--", c="r", lw=1)
    for t in TARGETS: ax.axvline(t, ls=":", c="k", lw=.8)
    ax.set_xlabel("고정한 민감도"); ax.set_ylabel("그때의 특이도")
    ax.set_title(f"{pname} — 재현율 우선 동작점의 대가", fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.tight_layout(); run.save_fig("operating_point", fig); plt.show()

run.save_json("evaluation", {"operating_points": OP, "fbeta": FB, "ppv_npv": NPV,
                             "G0": g0_ok, "P-1": P1, "P-2": P2, "P-3": P3,
                             "verdict": verdict})

result = {"week": 2, "exp_id": "exp14_clinical_op", "quest": "ailab-2026-0015",
          "task": "재현율 우선 동작점(R1)이 현실적인 요구인지 학습 없이 잰다",
          "split": "inter", "metric": "specificity_at_sens090_12lead",
          "value": round(p1_m["spec"], 4), "passed": bool(P1 is True and P2),
          "date": time.strftime("%Y-%m-%d"), "training": "없음(실험10′ OOF 재분석)",
          "caveat": CONFIG["caveat"], "operating_points": OP, "fbeta": FB,
          "ppv_npv": NPV, "G0": g0_ok, "P-1": P1, "P-2": P2, "P-3": P3,
          "verdict": verdict,
          "summary": (f"MI 민감도 0.90 → {{12}} 특이도 {p1_m['spec']:.3f} "
                      f"[{p1_m['spec_ci'][0]:.3f},{p1_m['spec_ci'][1]:.3f}] · "
                      f"{{I,II}} {mi[KEY(0.90,'I+II')]['spec']:.3f}(차 {gap:+.3f}) · "
                      f"낙관편차 {opt:+.4f} · {verdict.split(' —')[0]}")}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp14_clinical_operating_point.ipynb \\
      --quest ailab-2026-0015 --step "exp14-clinical-operating-point" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")

---

## 결과 읽는 법

| P-1 | 뜻 | 실험15에 미치는 영향 |
|---|---|---|
| ✅ | 민감도 0.90을 고정해도 특이도가 버틴다 | **동작점 조정만으로 충분.** 손실함수를 건드리지 않는다 |
| ❌ | 재현율을 올리면 특이도가 무너진다 | 가중 손실(`pos_weight`) → Focal 순으로 **하나씩** 적용 (R2) |
| ⚠️ | CI가 하한을 걸친다 | 동작점 우선, 손실가중은 대안으로 준비 |

**P-3(낙관 편차)이 크면 그게 더 중요한 결과다.** 앞으로 어떤 임계값 보고도 교차적합
없이는 못 믿는다는 뜻이고, 논문·규제 문서의 방법론 절에 그대로 들어간다.

## 한계 — 표에 반드시 붙여 낸다

- **PTB-XL의 MI는 급성이 아니다.** 대부분 경색 패턴(병적 Q파)이고 내원시각·트로포닌이 없다.
  여기서 나온 특이도를 "응급실 배제 성능"으로 인용하면 거짓이 된다.
- **가정 유병률의 NPV는 가정이다.** PTB-XL 자체 유병률(≈0.16)은 응급실 흉통 코호트와 다르다.
  표의 NPV는 "만약 유병률이 p라면"이라는 조건부 진술이다.
- **모델은 소견을 학습한 적이 없다.** 여기서 재는 것은 MI *superclass* 검출이고,
  "하벽경색 민감도"는 실험15 이후에야 말할 수 있다.
- **음성군에 다른 이상이 섞여 있다**(CD·STTC·HYP). 정상만 음성으로 놓으면 특이도가 더 좋게
  나오지만 그건 임상 상황이 아니다 — 일부러 어려운 쪽으로 정의했다.
- 임계값은 **확률의 절대값**이라 모델을 재학습하면 옮겨간다. 옮기는 절차(교차적합)가
  자산이지 숫자 자체가 자산이 아니다.
